# 09 — Prithvi-EO-2.0-300M-TL: Location Embedding Ablation
**Runtime:** Google Colab — **GPU (T4 minimum, A100 preferred)**

**Purpose:** Ablation study comparing `prithvi_eo_v2_300` (base, no location/temporal
embeddings) vs `prithvi_eo_v2_300_tl` (transfer-learning variant **with** learned
location and temporal embeddings).

The TL variant adds sinusoidal location (lat/lon) and temporal (year, day-of-year)
embeddings as a learned weighted sum on the ViT token embeddings during pre-training.
This notebook tests whether those embeddings improve downstream connectivity prediction
when the encoder is frozen.

Three heads are evaluated — matching NB04, NB05, and the Prithvi-only branch of NB06:

| Head | Architecture | Reference |
|---|---|---|
| Linear | LayerNorm(1024) → Dropout → Linear(1) | NB04 |
| MLP | LayerNorm → Linear→GELU→Dropout ×2 → Linear(1) | NB05 |
| XGBoost | 48-config random search | NB06 (Prithvi-only branch) |

**Inputs:**
- `data/raw/patches_2019/` — 6,970 GeoTIFF patches (224×224×6 HLS bands)
- `data/processed/patches_with_splits.csv` — patch metadata + labels

## Step 0: Environment Setup

> **Note:** The cell below installs `terratorch` and pins NumPy.  
> After it runs the runtime will **restart automatically**.  
> Re-run from **Step 0.2** after the restart.

In [ ]:
# Cell 0.1: Install dependencies
!pip install -q "numpy==1.26.4"
!pip install -q terratorch
!pip install -q xgboost rasterio scikit-learn geopandas matplotlib seaborn scipy pyarrow joblib

import os
print("Restarting runtime to apply numpy pin — re-run from Step 0.2 after restart.")
os.kill(os.getpid(), 9)

In [ ]:
# Cell 0.2: Clone repo
import os

REPO = 'satellite-internet-prediction-ML'
if not os.getcwd().endswith(REPO):
    if os.path.exists(REPO):
        %cd {REPO}
    else:
        !git clone https://github.com/tatyana21111/{REPO}.git
        %cd {REPO}
!git pull

In [ ]:
# Cell 0.3: Sync patches from Google Drive
from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

PATCH_DIR = 'data/raw/patches_2019'
Path(PATCH_DIR).mkdir(parents=True, exist_ok=True)

local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
if local_count < 6900:
    print('Syncing patches from Google Drive ...')
    drive_dir = '/content/drive/MyDrive/patches_2019'
    for f in Path(drive_dir).glob('*.tif'):
        dst = Path(PATCH_DIR) / f.name
        if not dst.exists():
            shutil.copy(f, dst)
    local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
print(f'Patches available: {local_count:,}')

patch_count = len(list(Path(PATCH_DIR).glob('*.tif')))
if patch_count < 6900:
    raise FileNotFoundError(
        f'Only {patch_count} patches found in {PATCH_DIR}. '
        f'Run NB01 first (Steps 1-7) to download patches and save them to Google Drive.'
    )

## Step 1: Imports & Constants

In [ ]:
import random, numpy as np, torch
random.seed(42); np.random.seed(42)
torch.manual_seed(42); torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True; torch.use_deterministic_algorithms(True)

import pandas as pd
import rasterio
import torch.nn as nn
import pytorch_lightning as pl
import matplotlib.pyplot as plt
import xgboost as xgb
import warnings, logging, json
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    roc_auc_score, f1_score, precision_score,
    recall_score, accuracy_score, mean_absolute_error, mean_squared_error,
)
from sklearn.model_selection import ParameterSampler, StratifiedKFold
from scipy.stats import spearmanr
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
logging.getLogger('rasterio').setLevel(logging.ERROR)

HLS_MEANS = [0.14245495, 0.13921481, 0.12434631, 0.31420089, 0.20743526, 0.12046503]
HLS_STDS  = [0.04036231, 0.04186983, 0.05267646, 0.08222210, 0.06834774, 0.05294205]
SCALE     = 0.0001

PATCH_DIR   = 'data/raw/patches_2019'
RESULTS_DIR = 'outputs/results'
FIGURES_DIR = 'outputs/figures'
for d in [RESULTS_DIR, FIGURES_DIR, 'outputs/models', 'checkpoints']:
    Path(d).mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## Step 2: Load Data

In [ ]:
sampled = pd.read_csv('data/processed/patches_with_splits.csv')

available = set(f.stem for f in Path(PATCH_DIR).glob('*.tif'))
sampled = sampled[sampled['patch_id'].isin(available)].reset_index(drop=True)

train_df = sampled[sampled['split'] == 'train'].reset_index(drop=True)
val_df   = sampled[sampled['split'] == 'val'].reset_index(drop=True)
test_df  = sampled[sampled['split'] == 'test'].reset_index(drop=True)

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test (NE): {len(test_df):,}')
print(f'Test binary positive rate: {test_df["has_coverage"].mean():.2%}')

## Step 3: Extract Prithvi-300M-TL Embeddings (with Location + Temporal)

Uses `prithvi_eo_v2_300_tl` which was **pre-trained with** learned location and temporal
embedding weights. We pass each patch’s centre (lat, lon) and a fixed temporal coordinate
(year=2019, day-of-year=45 for the Q1 HLS composite).

The resulting 1024-d embeddings encode both spectral-spatial content **and** geographic
position, allowing downstream heads to exploit location-aware representations.

In [ ]:
OUTPUT_FEATURES = Path('outputs/features')
OUTPUT_FEATURES.mkdir(parents=True, exist_ok=True)

DRIVE_FEATURES = Path('/content/drive/MyDrive/prithvi_embeddings')
DRIVE_FEATURES.mkdir(parents=True, exist_ok=True)

TEMPORAL_COORD = [2019, 45]  # year=2019, day-of-year=45 (mid-Feb, HLS Q1 composite)


class PrithviTLPatchDataset(Dataset):
    """Dataset that returns image tensor + (lat, lon) for each patch."""
    def __init__(self, df, patch_dir, n_bands=6):
        self.df = df.reset_index(drop=True)
        self.patch_dir = patch_dir
        self.n_bands = n_bands

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with rasterio.open(f"{self.patch_dir}/{row['patch_id']}.tif") as src:
            img = src.read(list(range(1, self.n_bands + 1))).astype(np.float32)
        img *= SCALE
        for b in range(self.n_bands):
            img[b] = (img[b] - HLS_MEANS[b]) / HLS_STDS[b]
        img = np.nan_to_num(img, nan=0.0)
        # (C, H, W) -> (C, T=1, H, W)
        img_tensor = torch.from_numpy(img).unsqueeze(1)
        loc = torch.tensor([row['lat'], row['lon']], dtype=torch.float32)
        return img_tensor, loc, row['patch_id']


def load_or_extract_tl_embeddings(split_name, df):
    """Load cached Prithvi-300M-TL embeddings, or extract + cache them.
    These are DIFFERENT from the base model embeddings (prithvi_frozen_embeds_*)."""
    local_path = OUTPUT_FEATURES / f'prithvi_tl_embeds_{split_name}.npz'
    drive_path = DRIVE_FEATURES / f'prithvi_tl_embeds_{split_name}.npz'

    if local_path.exists():
        data = np.load(local_path)
        X = data['X'].astype(np.float32)
        print(f'  Loaded cached {split_name} (local): {X.shape}')
        return X

    if drive_path.exists():
        import shutil
        shutil.copy(drive_path, local_path)
        data = np.load(local_path)
        X = data['X'].astype(np.float32)
        print(f'  Loaded cached {split_name} (Drive): {X.shape}')
        return X

    print(f'  Cache miss for {split_name} \u2014 extracting with prithvi_eo_v2_300_tl ...')
    from terratorch.registry import BACKBONE_REGISTRY

    encoder = BACKBONE_REGISTRY.build('prithvi_eo_v2_300_tl', pretrained=True).to(DEVICE).eval()
    for p in encoder.parameters():
        p.requires_grad = False

    ds = PrithviTLPatchDataset(df, PATCH_DIR)
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    all_embs = []
    with torch.no_grad():
        for x, locs, _ in tqdm(loader, desc=f'Extracting {split_name}'):
            x = x.to(DEVICE, dtype=torch.float32)
            B = x.shape[0]
            loc_coords = locs.to(DEVICE, dtype=torch.float32)                      # (B, 2)
            temp_coords = torch.tensor([TEMPORAL_COORD], dtype=torch.float32,
                                       device=DEVICE).unsqueeze(0).expand(B, 1, 2)  # (B, T=1, 2)
            feats = encoder(x, temporal_coords=temp_coords, location_coords=loc_coords)
            pooled = feats[-1].mean(dim=1)
            all_embs.append(pooled.cpu().numpy().astype(np.float32))

    X = np.concatenate(all_embs, axis=0)
    np.savez_compressed(local_path, X=X, patch_id=df['patch_id'].to_numpy())
    print(f'  Extracted and cached {split_name}: {X.shape}')

    import shutil
    shutil.copy(local_path, drive_path)
    print(f'  Saved to Drive: {drive_path}')

    del encoder
    torch.cuda.empty_cache()
    return X


print('Loading frozen Prithvi-300M-TL embeddings (1024-d, with location+temporal) ...')
X_train_emb = load_or_extract_tl_embeddings('train', train_df)
X_val_emb   = load_or_extract_tl_embeddings('val',   val_df)
X_test_emb  = load_or_extract_tl_embeddings('test',  test_df)
print(f'\nTrain: {X_train_emb.shape}  |  Val: {X_val_emb.shape}  |  Test: {X_test_emb.shape}')

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, targets):
        self.X = torch.from_numpy(embeddings)
        self.y = torch.tensor(targets, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## Step 4: Shared Evaluation Function

In [ ]:
def evaluate_model(test_preds, y_test_cont, y_test_bin, val_preds, y_val_bin,
                   model_name, target_name):
    """Compute all metrics and return a dict. Finds val-optimal binarisation threshold."""
    prec_v, rec_v, thr_v = precision_recall_curve(y_val_bin, val_preds)
    f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
    opt_thr = float(thr_v[np.argmax(f1_v)])

    y_pred_bin = (test_preds >= opt_thr).astype(int)
    mae  = mean_absolute_error(y_test_cont, test_preds)
    rmse = np.sqrt(mean_squared_error(y_test_cont, test_preds))
    rho, _ = spearmanr(y_test_cont, test_preds)

    pr_auc   = average_precision_score(y_test_bin, test_preds)
    roc_auc  = roc_auc_score(y_test_bin, test_preds)
    f1_opt   = f1_score(y_test_bin, y_pred_bin)
    prec_opt = precision_score(y_test_bin, y_pred_bin, zero_division=0)
    rec_opt  = recall_score(y_test_bin, y_pred_bin)
    acc_opt  = accuracy_score(y_test_bin, y_pred_bin)

    metrics = {
        'model': model_name, 'target': target_name,
        'mae': mae, 'rmse': rmse, 'spearman_rho': rho,
        'pr_auc': pr_auc, 'roc_auc': roc_auc, 'f1_opt': f1_opt,
        'opt_threshold': opt_thr,
        'precision_opt': prec_opt, 'recall_opt': rec_opt, 'accuracy_opt': acc_opt,
        'n_train': len(train_df) + len(val_df), 'n_test': len(test_df),
    }
    print(f'  RMSE={rmse:.4f}  Spearman={rho:.4f}  PR-AUC={pr_auc:.4f}  '
          f'ROC-AUC={roc_auc:.4f}  F1={f1_opt:.4f}')
    return metrics

## Step 5: Head A — Linear Probe
Matches NB04 architecture: `LayerNorm(1024) → Dropout(0.1) → Linear(1024, 1)`

In [ ]:
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint


class LinearRegressor(pl.LightningModule):
    def __init__(self, target_col, lr=1e-3, weight_decay=1e-3, dropout=0.1):
        super().__init__()
        self.save_hyperparameters()
        self.head = nn.Sequential(
            nn.LayerNorm(1024),
            nn.Dropout(dropout),
            nn.Linear(1024, 1),
        )
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        return self.head(x).squeeze(1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        self.log('val_loss', self.loss_fn(self(x), y), prog_bar=True, sync_dist=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr,
                                weight_decay=self.hparams.weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sched, 'monitor': 'val_loss'}}


class MLPRegressor(pl.LightningModule):
    def __init__(self, target_col, lr=3e-4, weight_decay=1e-3, dropout=0.2,
                 hidden_dim_1=256, hidden_dim_2=128):
        super().__init__()
        self.save_hyperparameters()
        self.head = nn.Sequential(
            nn.LayerNorm(1024),
            nn.Linear(1024, hidden_dim_1), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim_1, hidden_dim_2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim_2, 1),
        )
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        return self.head(x).squeeze(1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        self.log('val_loss', self.loss_fn(self(x), y), prog_bar=True, sync_dist=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr,
                                weight_decay=self.hparams.weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sched, 'monitor': 'val_loss'}}

In [ ]:
def train_pl_head(HeadClass, head_name, head_kwargs, targets, max_epochs=100, patience=15, bs=32):
    """Train a PL head on both targets, return metrics dict."""
    all_metrics = {}
    for target in targets:
        print(f'\n{"="*60}')
        print(f'  {head_name} \u2014 {target}')
        print(f'{"="*60}')

        train_loader = DataLoader(
            EmbeddingDataset(X_train_emb, train_df[target].values),
            batch_size=bs, shuffle=True, num_workers=0, pin_memory=True)
        val_loader = DataLoader(
            EmbeddingDataset(X_val_emb, val_df[target].values),
            batch_size=bs * 2, shuffle=False, num_workers=0, pin_memory=True)

        model = HeadClass(target_col=target, **head_kwargs)
        early_stop = EarlyStopping(monitor='val_loss', patience=patience, mode='min', verbose=True)
        ckpt_cb = ModelCheckpoint(
            dirpath='checkpoints',
            filename=f'prithvi_tl_{head_name}_{target}_' + '{epoch:02d}_{val_loss:.4f}',
            monitor='val_loss', mode='min', save_top_k=1, verbose=True)

        trainer = Trainer(
            max_epochs=max_epochs,
            accelerator='gpu' if torch.cuda.is_available() else 'cpu', devices=1,
            callbacks=[early_stop, ckpt_cb], log_every_n_steps=10, enable_progress_bar=True)
        trainer.fit(model, train_loader, val_loader)
        print(f'Best val loss: {ckpt_cb.best_model_score:.6f}')

        best_model = HeadClass.load_from_checkpoint(ckpt_cb.best_model_path).to(DEVICE).eval()

        def _infer(mdl, embs, tgts):
            ds = EmbeddingDataset(embs, tgts)
            loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=0)
            preds = []
            mdl.eval()
            with torch.no_grad():
                for x, _ in loader:
                    preds.extend(mdl(x.to(DEVICE)).cpu().numpy())
            return np.array(preds)

        val_preds  = _infer(best_model, X_val_emb, val_df[target].values)
        test_preds = _infer(best_model, X_test_emb, test_df[target].values)

        m = evaluate_model(
            test_preds, test_df[target].values, test_df['has_coverage'].values,
            val_preds, val_df['has_coverage'].values,
            model_name=f'Prithvi_300M_TL_{head_name}_{target}', target_name=target)
        all_metrics[target] = m

        del best_model, trainer, model
        torch.cuda.empty_cache()

    return all_metrics

In [ ]:
TARGETS = ['coverage_fraction', 'log_mean_tests']

print('\n' + '='*70)
print('  HEAD A: Linear Probe (matches NB04 config)')
print('='*70)
linear_metrics = train_pl_head(
    LinearRegressor, 'linear',
    dict(lr=1e-3, weight_decay=1e-3, dropout=0.1),
    TARGETS, max_epochs=100, patience=15, bs=32)

## Step 6: Head B — MLP (matches NB05 config)

In [ ]:
print('\n' + '='*70)
print('  HEAD B: MLP (matches NB05 config)')
print('='*70)
mlp_metrics = train_pl_head(
    MLPRegressor, 'mlp',
    dict(lr=3e-4, weight_decay=1e-3, dropout=0.2, hidden_dim_1=256, hidden_dim_2=128),
    TARGETS, max_epochs=150, patience=15, bs=32)

## Step 7: Head C — XGBoost (48-config random search, matches NB06 Prithvi-only)

In [ ]:
PARAM_SPACE = {
    'n_estimators':     [300, 600, 1200],
    'max_depth':        [3, 5, 7, 9],
    'learning_rate':    [0.01, 0.03, 0.05, 0.1],
    'subsample':        [0.6, 0.8, 1.0],
    'colsample_bytree': [0.5, 0.7, 1.0],
    'reg_alpha':        [0, 0.1, 1.0],
    'reg_lambda':       [1.0, 3.0, 10.0],
    'min_child_weight': [1, 5, 10],
}

N_CONFIGS = 48
configs = list(ParameterSampler(PARAM_SPACE, n_iter=N_CONFIGS, random_state=42))
print(f'XGBoost random search: {N_CONFIGS} configurations')

xgb_metrics = {}

X_train_val = np.vstack([X_train_emb, X_val_emb])

for target in TARGETS:
    print(f'\n{"="*60}')
    print(f'  XGBoost \u2014 {target}')
    print(f'{"="*60}')

    y_train_val = np.concatenate([train_df[target].values, val_df[target].values])
    y_bin_tv    = np.concatenate([train_df['has_coverage'].values, val_df['has_coverage'].values])

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    folds = list(skf.split(X_train_val, y_bin_tv))

    best_score, best_cfg = 1e9, None
    for i, cfg in enumerate(configs):
        fold_scores = []
        for tr_idx, va_idx in folds:
            mdl = xgb.XGBRegressor(
                **cfg, tree_method='hist', device='cuda',
                objective='reg:squarederror', random_state=42,
                early_stopping_rounds=30)
            mdl.fit(X_train_val[tr_idx], y_train_val[tr_idx],
                    eval_set=[(X_train_val[va_idx], y_train_val[va_idx])],
                    verbose=False)
            preds = mdl.predict(X_train_val[va_idx])
            fold_scores.append(np.sqrt(mean_squared_error(y_train_val[va_idx], preds)))
        mean_rmse = np.mean(fold_scores)
        if mean_rmse < best_score:
            best_score = mean_rmse
            best_cfg = cfg
        if (i + 1) % 12 == 0:
            print(f'  [{i+1}/{N_CONFIGS}] best CV RMSE so far: {best_score:.4f}')

    print(f'  Best config: {best_cfg}')
    print(f'  Best CV RMSE: {best_score:.4f}')

    final_model = xgb.XGBRegressor(
        **best_cfg, tree_method='hist', device='cuda',
        objective='reg:squarederror', random_state=42,
        early_stopping_rounds=30)
    final_model.fit(
        X_train_emb, train_df[target].values,
        eval_set=[(X_val_emb, val_df[target].values)],
        verbose=False)

    model_path = f'outputs/models/prithvi_tl_xgboost_{target}.json'
    final_model.save_model(model_path)
    print(f'  Saved: {model_path}')

    val_preds  = final_model.predict(X_val_emb)
    test_preds = final_model.predict(X_test_emb)

    m = evaluate_model(
        test_preds, test_df[target].values, test_df['has_coverage'].values,
        val_preds, val_df['has_coverage'].values,
        model_name=f'Prithvi_300M_TL_xgboost_{target}', target_name=target)
    xgb_metrics[target] = m

## Step 8: Comparison — Base vs TL (Location-Embedded)

Load NB04/NB05 results (base Prithvi-300M without location embeddings) and compare
against the TL results from this notebook.

In [ ]:
rows = []

base_files = {
    ('Linear',  'NB04'): 'prithvi_linear_agg_{target}_metrics.csv',
    ('MLP',     'NB05'): 'prithvi_mlp_agg_{target}_metrics.csv',
    ('XGBoost', 'NB06'): 'fusion_prithvi_only_{target}_metrics.csv',
}

for target in TARGETS:
    for (head, nb), pattern in base_files.items():
        p = Path(RESULTS_DIR) / pattern.format(target=target)
        if p.exists():
            b = pd.read_csv(p).iloc[0]
            rows.append({
                'Model': f'Base {head} ({nb})',
                'Target': target,
                'RMSE': b.get('rmse', float('nan')),
                'Spearman': b.get('spearman_rho', float('nan')),
                'PR-AUC': b.get('pr_auc', float('nan')),
                'ROC-AUC': b.get('roc_auc', float('nan')),
                'F1': b.get('f1_opt', float('nan')),
            })
        else:
            print(f'  (not found: {p})')

    for head, m_dict in [('Linear', linear_metrics), ('MLP', mlp_metrics), ('XGBoost', xgb_metrics)]:
        m = m_dict[target]
        rows.append({
            'Model': f'TL {head} (NB09)',
            'Target': target,
            'RMSE': m['rmse'],
            'Spearman': m['spearman_rho'],
            'PR-AUC': m['pr_auc'],
            'ROC-AUC': m['roc_auc'],
            'F1': m['f1_opt'],
        })

comp_df = pd.DataFrame(rows)

for target in TARGETS:
    print(f'\n{"="*60}')
    print(f'  {target}')
    print(f'{"="*60}')
    sub = comp_df[comp_df['Target'] == target]
    print(sub[['Model', 'RMSE', 'Spearman', 'PR-AUC', 'ROC-AUC', 'F1']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Prithvi-300M Base vs TL (Location+Temporal Embeddings)\nNE India Test Set',
             fontsize=14, fontweight='bold')

for ax_i, target in enumerate(TARGETS):
    ax = axes[ax_i]
    sub = comp_df[comp_df['Target'] == target].reset_index(drop=True)
    x = np.arange(len(sub))
    w = 0.18

    for j, (metric, color) in enumerate([
        ('RMSE', '#e74c3c'), ('Spearman', '#3498db'),
        ('PR-AUC', '#2ecc71'), ('ROC-AUC', '#f39c12'), ('F1', '#9b59b6')
    ]):
        vals = sub[metric].fillna(0).values
        ax.bar(x + j * w, vals, w, label=metric, color=color, alpha=0.85)

    ax.set_xticks(x + 2 * w)
    ax.set_xticklabels(sub['Model'], rotation=25, ha='right', fontsize=9)
    ax.set_title(target, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.1)
    if ax_i == 0:
        ax.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/prithvi_tl_vs_base_comparison.png', dpi=200, bbox_inches='tight')
print(f'Saved: {FIGURES_DIR}/prithvi_tl_vs_base_comparison.png')
plt.show()

## Step 9: Save Metrics & Push to GitHub

In [ ]:
for head, m_dict in [('linear', linear_metrics), ('mlp', mlp_metrics), ('xgboost', xgb_metrics)]:
    for target, m in m_dict.items():
        pd.DataFrame([m]).to_csv(
            f'{RESULTS_DIR}/prithvi_tl_{head}_{target}_metrics.csv', index=False)

comp_df.to_csv(f'{RESULTS_DIR}/prithvi_tl_vs_base_comparison.csv', index=False)
print('All metrics saved.')

In [ ]:
import subprocess, os

token = "YOUR_TOKEN_HERE"
repo  = "tatyana21111/satellite-internet-prediction-ML"

subprocess.run(['git', 'config', '--global', 'user.email', 'tatyana211zy@outlook.com'], check=True)
subprocess.run(['git', 'config', '--global', 'user.name',  'tatyana21111'], check=True)
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/{repo}.git'], check=True)

subprocess.run(['git', 'add', '-f', 'notebooks/09_prithvi_location_embedding.ipynb'], check=True)

for pat in ['prithvi_tl_*.csv', 'prithvi_tl_vs_base_comparison.csv']:
    for p in Path(RESULTS_DIR).glob(pat):
        subprocess.run(['git', 'add', '-f', str(p)], check=True)
        print(f'  Staged: {p}')

for p in Path(FIGURES_DIR).glob('prithvi_tl_*.png'):
    subprocess.run(['git', 'add', '-f', str(p)], check=True)
    print(f'  Staged: {p}')

diff = subprocess.run(['git', 'diff', '--cached', '--quiet'], capture_output=True)
if diff.returncode != 0:
    subprocess.run(['git', 'commit', '-m',
                    'NB09: Prithvi-300M-TL location embedding ablation \u2014 Linear, MLP, XGBoost'], check=True)
else:
    print('Nothing staged.')

subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'], check=True)
subprocess.run(['git', 'push'], check=True)
print('Push successful.')